## Example 1 — the installed version, and what is exported

In [ ]:
import importlib.metadata
import pydantic_monty

assert pydantic_monty.__version__ == importlib.metadata.version("pydantic_monty_client"), pydantic_monty.__version__
missing = [name for name in pydantic_monty.__all__ if not hasattr(pydantic_monty, name)]
assert not missing, missing
errors = (pydantic_monty.MontySyntaxError, pydantic_monty.MontyRuntimeError, pydantic_monty.MontyTypingError,
          pydantic_monty.MontyCrashedError)
assert all(issubclass(error, pydantic_monty.MontyError) for error in errors)
print("pydantic_monty", pydantic_monty.__version__, "exports", len(pydantic_monty.__all__), "names")

## Example 2 — where the worker program is looked for

In [ ]:
import os
import shutil
from pydantic_monty._binary import find_monty_binary

assert find_monty_binary("/opt/elsewhere/monty") == "/opt/elsewhere/monty"      # an explicit path wins
previous = os.environ.get("MONTY_BIN")
os.environ["MONTY_BIN"] = "/from/the/environment/monty"
try:
    assert find_monty_binary() == "/from/the/environment/monty"                  # then the MONTY_BIN variable
finally:
    os.environ.pop("MONTY_BIN") if previous is None else os.environ.__setitem__("MONTY_BIN", previous)
on_path = shutil.which("monty")
if on_path is None:
    print("SKIP: the `monty` worker program (PyPI package `pydantic-monty-runtime`) is not installed -> no sandbox run here")
else:
    found = find_monty_binary()
    assert os.path.isfile(found) and os.access(found, os.X_OK), found
    print("worker program found at", found)

## Example 3 — a sandbox session: results, state kept between snippets, inputs, a host function

In [ ]:
import shutil
from pydantic_monty import Monty

if shutil.which("monty") is None:
    print("SKIP: the `monty` worker program (PyPI package `pydantic-monty-runtime`) is not installed -> no sandbox run here")
else:
    calls = []
    def get_weather(city):
        calls.append(city)
        return {"city": city, "temp_c": 21}
    with Monty() as pool, pool.checkout() as session:
        assert session.feed_run("1 + 1") == 2
        session.feed_run("def square(n):\n    return n * n\ntotal = sum(square(i) for i in range(10))")
        assert session.feed_run("total") == 285                                   # globals persist in the session
        assert session.feed_run("[x * 2 for x in data]", inputs={"data": [1, 2, 3]}) == [2, 4, 6]
        code = 'report = get_weather("Paris")\nreport["temp_c"] + 1'
        assert session.feed_run(code, external_lookup={"get_weather": get_weather}) == 22 and calls == ["Paris"]
    print("sandbox session: 1 + 1 = 2, state kept, inputs bound, the host function was called with", calls)

## Example 4 — errors inside the sandbox come back as Monty errors, and the session survives

In [ ]:
import shutil
from pydantic_monty import CollectString, Monty, MontyRuntimeError, MontySyntaxError

if shutil.which("monty") is None:
    print("SKIP: the `monty` worker program (PyPI package `pydantic-monty-runtime`) is not installed -> no sandbox run here")
else:
    with Monty() as pool, pool.checkout() as session:
        printed = CollectString()
        session.feed_run('print("hello", 3)', print_callback=printed)
        assert printed.output == "hello 3\n", printed.output
        seen = []
        for code, error_type, inner in (("1 / 0", MontyRuntimeError, ZeroDivisionError), ("def (", MontySyntaxError, SyntaxError)):
            try:
                session.feed_run(code)
                raise AssertionError(f"{code!r} did not fail")
            except error_type as error:
                assert isinstance(error.exception(), inner), error.exception()
                seen.append(type(error.exception()).__name__)
        assert session.feed_run("40 + 2") == 42
    print("captured print output; raised", seen, "and the session kept working")

## Example 5 — the sandbox walls: no host files, no environment; and a missing worker program is a clear error

In [ ]:
import shutil
from pydantic_monty import Monty, MontyRuntimeError

if shutil.which("monty") is None:
    print("SKIP: the `monty` worker program (PyPI package `pydantic-monty-runtime`) is not installed -> no sandbox run here")
else:
    refused = []
    with Monty() as pool:
        with pool.checkout() as session:
            for code in ('open("/etc/hostname").read()', 'import os\nos.environ["HOME"]'):
                try:
                    session.feed_run(code)
                    raise AssertionError(f"the sandbox allowed {code!r}")
                except MontyRuntimeError as error:
                    refused.append(type(error.exception()).__name__)
    assert refused == ["PermissionError", "RuntimeError"], refused
    print("host file and environment refused:", refused)
try:
    with Monty(binary_path="/nonexistent/monty"):
        raise AssertionError("a missing worker program was accepted")
except RuntimeError as error:
    print("missing worker program:", error)